# Laboratorio: Uso de Grafos de Conocimiento para RAG

En este laboratorio, exploraremos cómo utilizar un grafo de conocimiento (Knowledge Graph) como fuente de datos para un modelo de Generación Aumentada por Recuperación (RAG). Esto permite que el modelo de lenguaje consulte datos estructurados y obtenga respuestas más precisas y contextualizadas.

## 1. Configuración del Entorno

In [ ]:
!pip install -r ../../requirements.txt
!pip install networkx matplotlib

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path='../../.env')

## 2. Creación del Grafo de Conocimiento

Un grafo de conocimiento representa entidades (nodos) y las relaciones entre ellas (aristas). Para este ejemplo, crearemos un grafo simple sobre personajes de la historia de la computación.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Crear un nuevo grafo dirigido
G = nx.DiGraph()

# Añadir nodos (entidades)
G.add_node("Ada Lovelace", type="persona")
G.add_node("Charles Babbage", type="persona")
G.add_node("Máquina Analítica", type="concepto")
G.add_node("Algoritmo", type="concepto")

# Añadir aristas (relaciones)
G.add_edge("Ada Lovelace", "Máquina Analítica", relation="escribió el primer algoritmo para")
G.add_edge("Charles Babbage", "Máquina Analítica", relation="diseñó")
G.add_edge("Ada Lovelace", "Algoritmo", relation="es considerada la primera programadora de")

### Visualización del Grafo

In [ ]:
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=3000, node_color='skyblue', font_size=10, font_weight='bold')
edge_labels = nx.get_edge_attributes(G, 'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
plt.title("Grafo de Conocimiento de Pioneros de la Computación")
plt.show()

## 3. Consulta del Grafo de Conocimiento

Ahora, crearemos una función para "consultar" nuestro grafo. Esta función buscará un nodo y devolverá la información conectada a él.

In [ ]:
def consultar_grafo(graph, entidad):
    """Consulta el grafo para una entidad y devuelve sus relaciones."""
    if entidad not in graph:
        return f"La entidad '{entidad}' no se encontró en el grafo."
    
    relaciones = []
    # Buscar relaciones donde la entidad es el origen
    for source, target, data in graph.out_edges(entidad, data=True):
        relaciones.append(f"{source} {data['relation']} {target}")
    
    # Buscar relaciones donde la entidad es el destino
    for source, target, data in graph.in_edges(entidad, data=True):
        relaciones.append(f"{source} {data['relation']} {target}")
            
    return ". ".join(relaciones) if relaciones else f"No se encontraron relaciones para '{entidad}'."

In [ ]:
# Ejemplo de consulta
consulta = "Máquina Analítica"
info = consultar_grafo(G, consulta)
print(f"Información sobre {consulta}: {info}")

## 4. Integración con un Modelo de Lenguaje (RAG)

Finalmente, usaremos la información del grafo para aumentar un prompt y obtener una respuesta más informada de un modelo de lenguaje.\n\n**Nota Importante:** El siguiente código utiliza `azure-ai-agents` para interactuar con un modelo de lenguaje. Para que funcione, debes tener un modelo desplegado (por ejemplo, GPT-4) y haber configurado tu entorno correctamente. Asegúrate de que la variable `AZURE_OPENAI_DEPLOYMENT_NAME` esté definida en tu archivo `.env` con el nombre de tu despliegue.

In [ ]:
from azure.ai.agents.experimental import Agent
from azure.ai.agents.experimental.chat import ChatHistory

def rag_con_grafo(pregunta, entidad):
    """Realiza una consulta RAG usando el grafo de conocimiento."""
    # 1. Recuperar información del grafo
    contexto = consultar_grafo(G, entidad)
    print(f"Contexto recuperado del grafo: {contexto}\n")
    
    # 2. Construir el prompt aumentado
    prompt = f"""
    Basado en el siguiente contexto, responde la pregunta.
    Contexto: {contexto}
    Pregunta: {pregunta}
    """
    print(f"Prompt enviado al modelo:\n{prompt}\n")
    
    # 3. Llamar al modelo de lenguaje (usando azure-ai-agents)
    model_name = os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME")
    if not model_name:
        return "Error: La variable de entorno AZURE_OPENAI_DEPLOYMENT_NAME no está configurada. Por favor, configúrala en el archivo .env."
    
    try:
        agent = Agent(model=model_name)
        history = ChatHistory()
        history.add_user_message(prompt)
        result = agent.run(history)
        return result['choices'][0]['message']['content']
    except Exception as e:
        error_message = f"No se pudo llamar al modelo de lenguaje. Error: {e}."
        simulated_response = "Este es un resultado simulado: Ada Lovelace fue una matemática y escritora, famosa por su trabajo en la Máquina Analítica de Charles Babbage."
        print(error_message)
        return simulated_response

In [ ]:
# Ejemplo de RAG
pregunta_final = "¿Qué hizo Ada Lovelace?"
entidad_a_buscar = "Ada Lovelace"

respuesta = rag_con_grafo(pregunta_final, entidad_a_buscar)
print(f"Respuesta del modelo: {respuesta}")